# HAM10000: CNN Baseline vs. VAE-Augmented Classification

Self-contained Kaggle notebook. Running the first code cells writes out the
full `src/` package to `/kaggle/working/src/`, so this notebook needs
nothing attached except the **HAM10000 dataset** itself.

**Before running:** click "Add Input" and attach
`kmader/skin-cancer-mnist-ham10000`.

**GPU quota reality check:** Kaggle's free tier gives ~30 GPU-hours/week
and a 12-hour session cap. This notebook is organized into clearly marked
**SESSION** sections -- run one session, save the version (which persists
`/kaggle/working/` as this notebook's output), and continue in a later
session. Trying to run everything (4 conditions x 3 seeds + 6 VAEs) in one
sitting will likely exceed the weekly quota.


In [ ]:
import os
os.makedirs('src', exist_ok=True)
for d in ['data/raw','data/processed','data/manifests','data/synthetic',
          'outputs/checkpoints','outputs/logs','outputs/figures','outputs/predictions',
          'configs']:
    os.makedirs(d, exist_ok=True)

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## Write out the project source files

Run this once per session (it's idempotent -- just overwrites the files).

In [ ]:
%%writefile src/config.py
"""
Central configuration for the HAM10000 CNN-baseline vs VAE-augmented experiment.
All paths are relative to the project root so the code runs unchanged on
Kaggle (where the root is typically /kaggle/working) or locally.
"""
import os

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
PROJECT_ROOT = os.environ.get("HAM10000_PROJECT_ROOT", os.getcwd())

DATA_RAW_DIR = os.path.join(PROJECT_ROOT, "data", "raw")
DATA_PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
DATA_MANIFEST_DIR = os.path.join(PROJECT_ROOT, "data", "manifests")
DATA_SYNTHETIC_DIR = os.path.join(PROJECT_ROOT, "data", "synthetic")

OUTPUT_CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, "outputs", "checkpoints")
OUTPUT_LOG_DIR = os.path.join(PROJECT_ROOT, "outputs", "logs")
OUTPUT_FIGURE_DIR = os.path.join(PROJECT_ROOT, "outputs", "figures")
OUTPUT_PREDICTION_DIR = os.path.join(PROJECT_ROOT, "outputs", "predictions")

# On Kaggle, the HAM10000 Kaggle mirror is mounted read-only here once the
# dataset is attached to the notebook:
#   /kaggle/input/skin-cancer-mnist-ham10000/
# Set this env var to override for local / other environments.
KAGGLE_INPUT_DIR = os.environ.get(
    "HAM10000_KAGGLE_INPUT",
    "/kaggle/input/skin-cancer-mnist-ham10000",
)

for _d in [
    DATA_RAW_DIR, DATA_PROCESSED_DIR, DATA_MANIFEST_DIR, DATA_SYNTHETIC_DIR,
    OUTPUT_CHECKPOINT_DIR, OUTPUT_LOG_DIR, OUTPUT_FIGURE_DIR, OUTPUT_PREDICTION_DIR,
]:
    os.makedirs(_d, exist_ok=True)

# ---------------------------------------------------------------------------
# Classes (HAM10000 diagnostic categories)
# ---------------------------------------------------------------------------
CLASS_NAMES = ["nv", "mel", "bkl", "bcc", "akiec", "vasc", "df"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}
NUM_CLASSES = len(CLASS_NAMES)

# Historically reported full-dataset counts (Tschandl et al. 2018). These are
# NOT hard-coded into the pipeline logic -- audit_data.py recomputes the real
# counts from the downloaded metadata CSV and everything downstream uses that.
REFERENCE_CLASS_COUNTS = {
    "nv": 6705, "mel": 1113, "bkl": 1099, "bcc": 514,
    "akiec": 327, "vasc": 142, "df": 115,
}
MINORITY_CLASSES = ["mel", "bkl", "bcc", "akiec", "vasc", "df"]  # everything but nv

# ---------------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------------
SEEDS = [17, 29, 41]

# Adapts to whatever machine this runs on (Kaggle gives 2-4 CPUs typically;
# avoids the "excessive worker creation" warning on constrained environments).
NUM_WORKERS = max(0, min(2, (os.cpu_count() or 1) - 1))

# ---------------------------------------------------------------------------
# Split ratios (grouped by lesion_id, see make_group_splits.py)
# ---------------------------------------------------------------------------
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
TEST_FRAC = 0.15

# ---------------------------------------------------------------------------
# Classifier (DenseNet-121) hyperparameters
# ---------------------------------------------------------------------------
CLF_IMAGE_SIZE = 224
CLF_BATCH_SIZE = 32
CLF_DROPOUT = 0.30
CLF_LABEL_SMOOTHING = 0.05
CLF_HEAD_LR = 3e-4
CLF_BACKBONE_LR = 3e-5
CLF_WEIGHT_DECAY = 1e-4
CLF_WARMUP_EPOCHS = 3
CLF_STAGE_A_EPOCHS = 3     # frozen backbone, head only
CLF_STAGE_B_MAX_EPOCHS = 30  # unfrozen fine-tuning, early stopping on val macro-F1
CLF_EARLY_STOP_PATIENCE = 6

# ---------------------------------------------------------------------------
# VAE hyperparameters
# ---------------------------------------------------------------------------
VAE_IMAGE_SIZE = 128
VAE_LATENT_DIM = 128
VAE_BATCH_SIZE = 32
VAE_LR = 2e-4
VAE_MAX_EPOCHS = 150
VAE_EARLY_STOP_PATIENCE = 15
VAE_BETA_START = 1e-4
VAE_BETA_END = 0.5   # lowered from 1.0: full KL weight over-penalizes and biases
                     # the decoder toward blurry "average" reconstructions,
                     # which is especially damaging on tiny classes (df, vasc)
                     # that already have little data to begin with.
VAE_BETA_WARMUP_EPOCHS = 20

# ---------------------------------------------------------------------------
# Balancing policy conditions
# ---------------------------------------------------------------------------
CONDITIONS = ["C0", "C1", "C2", "C3"]
CONDITION_DESCRIPTIONS = {
    "C0": "Original imbalanced real training set (baseline).",
    "C1": "Real images + classical oversampling / augmentation, no VAE.",
    "C2": "Real images + VAE synthetic images, each minority class raised "
          "to >=50% of the majority training count.",
    "C3": "Real images + VAE synthetic images, each class raised to the "
          "majority training count (fully equalized).",
}


In [ ]:
%%writefile src/audit_data.py
"""
Phase 3: Audit.

Verifies the downloaded HAM10000 metadata + images before anything is
fitted. Produces:
  - data/processed/audit.csv          (per-row validity flags)
  - data/processed/class_distribution.png
  - prints a summary report to stdout

Usage (on Kaggle, after attaching the "skin-cancer-mnist-ham10000" dataset):
    python src/audit_data.py \
        --metadata_csv /kaggle/input/skin-cancer-mnist-ham10000/HAM10000_metadata.csv \
        --image_dirs /kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_1 \
                     /kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_2
"""
import argparse
import hashlib
import os
import sys

import pandas as pd

sys.path.insert(0, os.path.dirname(__file__))
from config import CLASS_NAMES, DATA_PROCESSED_DIR, OUTPUT_FIGURE_DIR  # noqa: E402


def find_image_path(image_id: str, image_dirs):
    """HAM10000 ships images across two folders; locate whichever has it."""
    for d in image_dirs:
        for ext in (".jpg", ".jpeg", ".png"):
            candidate = os.path.join(d, image_id + ext)
            if os.path.isfile(candidate):
                return candidate
    return None


def file_md5(path, chunk_size=1 << 16):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def run_audit(metadata_csv: str, image_dirs, compute_hashes: bool = True):
    df = pd.read_csv(metadata_csv)

    required_cols = {"image_id", "dx", "lesion_id"}
    missing_cols = required_cols - set(df.columns)
    if missing_cols:
        raise ValueError(
            f"Metadata CSV is missing required columns: {missing_cols}. "
            f"Found columns: {list(df.columns)}"
        )

    has_patient_id = "patient_id" in df.columns or "lesion_id" in df.columns
    group_col = "patient_id" if "patient_id" in df.columns else "lesion_id"

    rows = []
    for _, row in df.iterrows():
        image_id = row["image_id"]
        label = row["dx"]
        lesion_id = row["lesion_id"]

        path = find_image_path(image_id, image_dirs)
        file_exists = path is not None
        label_valid = label in CLASS_NAMES

        rows.append({
            "image_id": image_id,
            "lesion_id": lesion_id,
            "patient_id": row.get("patient_id", lesion_id),
            "dx": label,
            "file_exists": file_exists,
            "label_valid": label_valid,
            "file_path": path,
            "md5": file_md5(path) if (compute_hashes and file_exists) else None,
        })

    audit_df = pd.DataFrame(rows)

    # Duplicate detection: same md5 hash appearing more than once.
    if compute_hashes:
        dup_counts = audit_df["md5"].value_counts()
        dup_hashes = dup_counts[dup_counts > 1].index
        audit_df["is_exact_duplicate_file"] = audit_df["md5"].isin(dup_hashes)
    else:
        audit_df["is_exact_duplicate_file"] = False

    audit_csv_path = os.path.join(DATA_PROCESSED_DIR, "audit.csv")
    audit_df.to_csv(audit_csv_path, index=False)

    # ---- Summary report ----
    n_total = len(audit_df)
    n_missing_files = (~audit_df["file_exists"]).sum()
    n_invalid_labels = (~audit_df["label_valid"]).sum()
    n_exact_dupes = audit_df["is_exact_duplicate_file"].sum()
    n_unique_lesions = audit_df["lesion_id"].nunique()
    n_unique_patients = audit_df["patient_id"].nunique() if "patient_id" in audit_df else n_unique_lesions
    multi_image_lesions = (audit_df.groupby("lesion_id").size() > 1).sum()

    class_counts = audit_df[audit_df["label_valid"]]["dx"].value_counts().reindex(CLASS_NAMES).fillna(0).astype(int)

    print("=" * 70)
    print("HAM10000 DATA AUDIT SUMMARY")
    print("=" * 70)
    print(f"Total metadata rows:            {n_total}")
    print(f"Missing image files:            {n_missing_files}")
    print(f"Invalid / unexpected labels:    {n_invalid_labels}")
    print(f"Exact duplicate files (by MD5): {n_exact_dupes}")
    print(f"Unique lesion_ids:              {n_unique_lesions}")
    print(f"Unique patient/group ids:       {n_unique_patients}  (grouping column: '{group_col}')")
    print(f"Lesions with >1 image:          {multi_image_lesions}  <-- leakage risk if split ignores this")
    print("-" * 70)
    print("Recomputed class distribution (from metadata, not hard-coded):")
    for c in CLASS_NAMES:
        print(f"  {c:6s}: {class_counts[c]:5d}")
    print("=" * 70)

    if n_missing_files > 0:
        print(f"WARNING: {n_missing_files} images referenced in metadata were not found "
              f"on disk. Check --image_dirs paths.")
    if multi_image_lesions > 0:
        print(f"NOTE: {multi_image_lesions} lesions have multiple images. "
              f"make_group_splits.py MUST split by '{group_col}', never by image_id, "
              f"to avoid train/test leakage.")

    # Class distribution plot
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt

        plt.figure(figsize=(8, 5))
        class_counts.plot(kind="bar")
        plt.title("HAM10000 class distribution (recomputed from metadata)")
        plt.ylabel("Image count")
        plt.xlabel("Diagnostic class")
        plt.tight_layout()
        fig_path = os.path.join(OUTPUT_FIGURE_DIR, "class_distribution.png")
        plt.savefig(fig_path, dpi=150)
        plt.close()
        print(f"Saved class distribution plot to: {fig_path}")
    except ImportError:
        print("matplotlib not available; skipped class distribution plot.")

    print(f"Saved full audit table to: {audit_csv_path}")
    return audit_df, group_col


def main():
    parser = argparse.ArgumentParser(description="Audit HAM10000 metadata + images.")
    parser.add_argument("--metadata_csv", required=True, help="Path to HAM10000_metadata.csv")
    parser.add_argument("--image_dirs", nargs="+", required=True,
                         help="One or more directories containing the .jpg images")
    parser.add_argument("--no_hashes", action="store_true",
                         help="Skip MD5 hashing (faster, but no duplicate detection)")
    args = parser.parse_args()

    run_audit(args.metadata_csv, args.image_dirs, compute_hashes=not args.no_hashes)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/make_group_splits.py
"""
Phase 4: Split.

Creates fixed, group-stratified train/val/test manifests so that no
lesion (or patient, if available) appears in more than one partition.
Runs once per seed in config.SEEDS and writes:
    data/manifests/splits_seed<seed>.csv

Each manifest row: image_id, lesion_id, patient_id, dx, label_idx,
file_path, partition, seed.

Usage:
    python src/make_group_splits.py --audit_csv data/processed/audit.csv
"""
import argparse
import os
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(__file__))
from config import (  # noqa: E402
    CLASS_TO_IDX, DATA_MANIFEST_DIR, SEEDS, TRAIN_FRAC, VAL_FRAC, TEST_FRAC,
)


def grouped_stratified_split(df: pd.DataFrame, group_col: str, label_col: str, seed: int,
                              train_frac=TRAIN_FRAC, val_frac=VAL_FRAC, test_frac=TEST_FRAC):
    """
    Assigns every group (lesion/patient) entirely to one partition, while
    ensuring EVERY class is represented in train/val/test proportionally
    (a per-class stratified group split), not just the overall totals.

    Naive global-capacity greedy allocation is a trap here: it satisfies the
    overall train/val/test size ratio but can starve val/test of minority
    classes entirely (verified by the smoke test on synthetic dummy data --
    see project notes). Splitting group lists independently within each
    class, then concatenating, fixes this and is the standard approach
    (equivalent in spirit to sklearn's StratifiedGroupKFold).
    """
    assert abs(train_frac + val_frac + test_frac - 1.0) < 1e-6
    from collections import defaultdict

    rng = np.random.RandomState(seed)

    # One representative label per group: the group's majority label.
    group_label = (
        df.groupby(group_col)[label_col]
        .agg(lambda s: s.value_counts().idxmax())
        .to_dict()
    )

    class_to_groups = defaultdict(list)
    for g, cls in group_label.items():
        class_to_groups[cls].append(g)

    group_to_partition = {}
    warnings = []

    for cls, cls_groups in class_to_groups.items():
        cls_groups = list(cls_groups)
        rng.shuffle(cls_groups)
        n = len(cls_groups)

        if n < 3:
            # Too few lesions/patients to split into 3 non-empty partitions
            # for this class -- put everything in train and warn loudly.
            # (This can genuinely happen for extreme minority classes and
            # must be surfaced, not silently hidden.)
            for g in cls_groups:
                group_to_partition[g] = "train"
            warnings.append(
                f"Class '{cls}' has only {n} group(s) -- ALL assigned to train. "
                f"This class cannot be evaluated on val/test with this grouping; "
                f"consider using a coarser group_col or flagging this as a limitation."
            )
            continue

        n_test = max(1, int(round(test_frac * n)))
        n_val = max(1, int(round(val_frac * n)))
        # keep at least 1 in train too
        n_val = min(n_val, n - n_test - 1) if (n - n_test - 1) >= 1 else max(0, n - n_test - 1)
        n_train = n - n_test - n_val

        for g in cls_groups[:n_train]:
            group_to_partition[g] = "train"
        for g in cls_groups[n_train:n_train + n_val]:
            group_to_partition[g] = "val"
        for g in cls_groups[n_train + n_val:]:
            group_to_partition[g] = "test"

    for w in warnings:
        print(f"WARNING: {w}")

    out = df.copy()
    out["partition"] = out[group_col].map(group_to_partition)
    out["label_idx"] = out[label_col].map(CLASS_TO_IDX)
    out["seed"] = seed
    return out


def report_split(split_df: pd.DataFrame, label_col: str):
    print("Partition sizes (images):")
    print(split_df["partition"].value_counts())
    print("\nPer-partition class distribution:")
    print(pd.crosstab(split_df["partition"], split_df[label_col]))


def main():
    parser = argparse.ArgumentParser(description="Create group-stratified HAM10000 splits.")
    parser.add_argument("--audit_csv", default=os.path.join("data", "processed", "audit.csv"))
    parser.add_argument("--group_col", default="lesion_id",
                         help="Use 'patient_id' if that column is reliable in your metadata, "
                              "otherwise 'lesion_id' (default, always available).")
    args = parser.parse_args()

    audit_df = pd.read_csv(args.audit_csv)
    valid_df = audit_df[audit_df["file_exists"] & audit_df["label_valid"]].copy()
    if audit_df["is_exact_duplicate_file"].any():
        # Keep only the first occurrence of each exact-duplicate file so a
        # duplicated image can't land in two different partitions.
        valid_df = valid_df.sort_values("image_id").drop_duplicates(subset="md5", keep="first")
        print(f"Dropped exact-duplicate files, kept first occurrence of each. "
              f"Remaining rows: {len(valid_df)}")

    for seed in SEEDS:
        split_df = grouped_stratified_split(valid_df, group_col=args.group_col,
                                             label_col="dx", seed=seed)
        out_path = os.path.join(DATA_MANIFEST_DIR, f"splits_seed{seed}.csv")
        split_df.to_csv(out_path, index=False)
        print(f"\n=== seed {seed} -> {out_path} ===")
        report_split(split_df, "dx")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/datasets.py
"""
PyTorch Dataset definitions.

- HAM10000ClassifierDataset: real (+ optionally synthetic) images for the
  DenseNet-121 classifier.
- SingleClassImageDataset: real training images for one class only, used to
  train that class's VAE.
"""
import os

import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms

from config import CLASS_TO_IDX, CLF_IMAGE_SIZE, VAE_IMAGE_SIZE

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def build_classifier_transforms(train: bool):
    if train:
        return transforms.Compose([
            transforms.Resize(256),
            transforms.RandomResizedCrop(CLF_IMAGE_SIZE, scale=(0.85, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    return transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(CLF_IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


def build_vae_transforms(train: bool = True):
    """
    train=True: adds flip + small rotation augmentation. Skin lesions have
    no canonical orientation, so this is legitimate augmentation of real
    images (not synthetic data) -- it meaningfully increases effective
    training diversity, which matters most for the smallest classes
    (df, vasc) where the VAE otherwise sees very few unique images.
    train=False (e.g. reconstruction-quality checks): no augmentation.
    """
    ops = [transforms.Resize((VAE_IMAGE_SIZE, VAE_IMAGE_SIZE))]
    if train:
        ops += [
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(20),
        ]
    ops += [transforms.ToTensor()]  # VAE decoder ends in sigmoid -> expects [0,1]
    return transforms.Compose(ops)


class HAM10000ClassifierDataset(Dataset):
    """
    real_df: dataframe with columns [file_path, dx] for one partition
             (train / val / test), already filtered to that partition.
    synthetic_df: optional dataframe with columns [file_path, dx] for
                  VAE-generated training images (train partition only --
                  never pass this for val/test).
    """

    def __init__(self, real_df: pd.DataFrame, train: bool, synthetic_df: pd.DataFrame = None):
        frames = [real_df[["file_path", "dx"]].copy()]
        if synthetic_df is not None and len(synthetic_df) > 0:
            frames.append(synthetic_df[["file_path", "dx"]].copy())
        self.df = pd.concat(frames, ignore_index=True)
        self.transform = build_classifier_transforms(train=train)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["file_path"]).convert("RGB")
        img = self.transform(img)
        label = CLASS_TO_IDX[row["dx"]]
        return img, torch.tensor(label, dtype=torch.long)


class SingleClassImageDataset(Dataset):
    """Training-partition images for exactly one class, for VAE training."""

    def __init__(self, file_paths, train: bool = True):
        self.file_paths = list(file_paths)
        self.transform = build_vae_transforms(train=train)

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        img = Image.open(self.file_paths[idx]).convert("RGB")
        img = self.transform(img)
        return img


In [ ]:
%%writefile src/models_densenet.py
"""
DenseNet-121 classifier builder, matching the project spec:
ImageNet-pretrained backbone + Dropout(0.30) + Linear(1024, num_classes).
"""
import torch.nn as nn
from torchvision.models import DenseNet121_Weights, densenet121

from config import CLF_DROPOUT, NUM_CLASSES


def build_densenet121(num_classes: int = NUM_CLASSES, pretrained: bool = True):
    weights = DenseNet121_Weights.DEFAULT if pretrained else None
    model = densenet121(weights=weights)
    model.classifier = nn.Sequential(
        nn.Dropout(p=CLF_DROPOUT),
        nn.Linear(model.classifier.in_features, num_classes),
    )
    return model


def freeze_backbone(model):
    """Stage A: freeze everything except the new classifier head."""
    for name, param in model.named_parameters():
        param.requires_grad = name.startswith("classifier")
    return model


def unfreeze_final_block(model):
    """
    Stage B: unfreeze the final dense block + transition layer + classifier.
    torchvision's DenseNet121 features are named:
    denseblock1..4, transition1..3, norm5. We unfreeze denseblock4 + norm5 + classifier.
    """
    for name, param in model.named_parameters():
        if (
            name.startswith("classifier")
            or "denseblock4" in name
            or "norm5" in name
        ):
            param.requires_grad = True
        else:
            param.requires_grad = False
    return model


def unfreeze_all(model):
    """Sensitivity-run only: unfreeze the entire backbone."""
    for param in model.parameters():
        param.requires_grad = True
    return model


def get_param_groups(model, head_lr, backbone_lr):
    """Discriminative learning rates: classifier head vs. rest of backbone."""
    head_params, backbone_params = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        (head_params if name.startswith("classifier") else backbone_params).append(param)
    groups = []
    if head_params:
        groups.append({"params": head_params, "lr": head_lr})
    if backbone_params:
        groups.append({"params": backbone_params, "lr": backbone_lr})
    return groups


In [ ]:
%%writefile src/models_vae.py
"""
Class-specific convolutional VAE, matching the project spec:
  Encoder: 3->32->64->128->256, each 4x4 stride-2 conv + BatchNorm + SiLU
  Latent: dim 128, two linear heads for mu / logvar
  Decoder: linear -> 256x8x8 -> transposed-conv 256->128->64->32->3, sigmoid
  Assumes 128x128 input (8x8 spatial after 4 stride-2 downsamples).
"""
import torch
import torch.nn as nn

from config import VAE_IMAGE_SIZE, VAE_LATENT_DIM

_DOWNSAMPLE_STEPS = 4
_FEATURE_SPATIAL = VAE_IMAGE_SIZE // (2 ** _DOWNSAMPLE_STEPS)  # 128 / 16 = 8


class ConvVAE(nn.Module):
    def __init__(self, latent_dim: int = VAE_LATENT_DIM):
        super().__init__()
        self.latent_dim = latent_dim

        def enc_block(cin, cout):
            return nn.Sequential(
                nn.Conv2d(cin, cout, kernel_size=4, stride=2, padding=1),
                nn.BatchNorm2d(cout),
                nn.SiLU(inplace=True),
            )

        self.encoder = nn.Sequential(
            enc_block(3, 32),
            enc_block(32, 64),
            enc_block(64, 128),
            enc_block(128, 256),
        )
        flat_dim = 256 * _FEATURE_SPATIAL * _FEATURE_SPATIAL
        self.fc_mu = nn.Linear(flat_dim, latent_dim)
        self.fc_logvar = nn.Linear(flat_dim, latent_dim)

        self.decoder_fc = nn.Linear(latent_dim, flat_dim)

        def dec_block(cin, cout, final=False):
            layers = [nn.ConvTranspose2d(cin, cout, kernel_size=4, stride=2, padding=1)]
            if not final:
                layers += [nn.BatchNorm2d(cout), nn.SiLU(inplace=True)]
            return nn.Sequential(*layers)

        self.decoder = nn.Sequential(
            dec_block(256, 128),
            dec_block(128, 64),
            dec_block(64, 32),
            dec_block(32, 3, final=True),
            nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.encoder(x)
        h = h.flatten(1)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = self.decoder_fc(z)
        h = h.view(-1, 256, _FEATURE_SPATIAL, _FEATURE_SPATIAL)
        return self.decoder(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z)
        return x_hat, mu, logvar

    @torch.no_grad()
    def sample(self, n: int, device):
        z = torch.randn(n, self.latent_dim, device=device)
        return self.decode(z).clamp(0.0, 1.0)


def vae_loss(x_hat, x, mu, logvar, beta):
    """L = reconstruction L1 + beta * KL(q(z|x) || N(0,I)), matching the spec."""
    recon = torch.nn.functional.l1_loss(x_hat, x, reduction="mean")
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    total = recon + beta * kl
    return total, recon.detach(), kl.detach()


def beta_schedule(epoch: int, warmup_epochs: int, beta_start: float, beta_end: float):
    if warmup_epochs <= 0:
        return beta_end
    frac = min(1.0, epoch / warmup_epochs)
    return beta_start + frac * (beta_end - beta_start)


In [ ]:
%%writefile src/train_vae.py
"""
Phase 6: Train one class-specific VAE.

Trains only on TRAIN-partition images of a single class (never val/test --
the frozen test/val partitions must never touch VAE training). Saves:
  outputs/checkpoints/vae_<class>_seed<seed>.pt
  outputs/figures/vae_<class>_seed<seed>_reconstructions.png
  outputs/figures/vae_<class>_seed<seed>_samples.png
  outputs/logs/vae_<class>_seed<seed>_losses.csv

Usage:
    python src/train_vae.py --manifest data/manifests/splits_seed17.csv \
        --class_name df --seed 17
"""
import argparse
import os
import sys

import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader

sys.path.insert(0, os.path.dirname(__file__))
from config import NUM_WORKERS  # noqa: E402
from config import (  # noqa: E402
    OUTPUT_CHECKPOINT_DIR, OUTPUT_FIGURE_DIR, OUTPUT_LOG_DIR,
    VAE_BATCH_SIZE, VAE_BETA_END, VAE_BETA_START, VAE_BETA_WARMUP_EPOCHS,
    VAE_EARLY_STOP_PATIENCE, VAE_LR, VAE_MAX_EPOCHS,
)
from datasets import SingleClassImageDataset  # noqa: E402
from models_vae import ConvVAE, beta_schedule, vae_loss  # noqa: E402


def save_image_grid(tensor_batch, path, nrow=5):
    from torchvision.utils import save_image
    save_image(tensor_batch, path, nrow=nrow)


def train_one_vae(manifest_path: str, class_name: str, seed: int,
                   max_epochs: int = VAE_MAX_EPOCHS, device=None):
    torch.manual_seed(seed)
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    df = pd.read_csv(manifest_path)
    train_df = df[(df["partition"] == "train") & (df["dx"] == class_name)]
    if len(train_df) < 10:
        raise ValueError(
            f"Only {len(train_df)} training images for class '{class_name}' -- "
            f"too few to train a VAE meaningfully. Check the manifest."
        )

    file_paths = train_df["file_path"].tolist()

    # Split by file path first (not via random_split of one Dataset) so
    # train and val get DIFFERENT transforms: train gets flip/rotation
    # augmentation, the small internal val set stays un-augmented for a
    # clean, comparable reconstruction-quality signal.
    rng_split = np.random.RandomState(seed)
    shuffled_paths = list(file_paths)
    rng_split.shuffle(shuffled_paths)
    n_val = max(1, int(0.1 * len(shuffled_paths)))
    val_paths = shuffled_paths[:n_val]
    train_paths = shuffled_paths[n_val:]

    train_ds = SingleClassImageDataset(train_paths, train=True)
    val_ds = SingleClassImageDataset(val_paths, train=False)

    train_loader = DataLoader(train_ds, batch_size=VAE_BATCH_SIZE, shuffle=True,
                               num_workers=NUM_WORKERS, drop_last=len(train_ds) > VAE_BATCH_SIZE)
    val_loader = DataLoader(val_ds, batch_size=VAE_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    model = ConvVAE().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=VAE_LR)

    best_val_loss = float("inf")
    epochs_no_improve = 0
    history = []

    ckpt_path = os.path.join(OUTPUT_CHECKPOINT_DIR, f"vae_{class_name}_seed{seed}.pt")

    for epoch in range(1, max_epochs + 1):
        beta = beta_schedule(epoch, VAE_BETA_WARMUP_EPOCHS, VAE_BETA_START, VAE_BETA_END)

        model.train()
        train_loss_sum, train_recon_sum, train_kl_sum, n_batches = 0.0, 0.0, 0.0, 0
        for x in train_loader:
            x = x.to(device)
            x_hat, mu, logvar = model(x)
            loss, recon, kl = vae_loss(x_hat, x, mu, logvar, beta)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss_sum += loss.item()
            train_recon_sum += recon.item()
            train_kl_sum += kl.item()
            n_batches += 1
        train_loss = train_loss_sum / max(1, n_batches)

        model.eval()
        val_loss_sum, val_batches = 0.0, 0
        with torch.no_grad():
            for x in val_loader:
                x = x.to(device)
                x_hat, mu, logvar = model(x)
                loss, _, _ = vae_loss(x_hat, x, mu, logvar, beta)
                val_loss_sum += loss.item()
                val_batches += 1
        val_loss = val_loss_sum / max(1, val_batches)

        history.append({
            "epoch": epoch, "beta": beta, "train_loss": train_loss,
            "train_recon": train_recon_sum / max(1, n_batches),
            "train_kl": train_kl_sum / max(1, n_batches), "val_loss": val_loss,
        })

        if epoch % 5 == 0 or epoch == 1:
            print(f"[{class_name} seed{seed}] epoch {epoch:3d}/{max_epochs} "
                  f"beta={beta:.4f} train_loss={train_loss:.4f} val_loss={val_loss:.4f}")

        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save({"model_state": model.state_dict(), "epoch": epoch,
                        "class_name": class_name, "seed": seed}, ckpt_path)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= VAE_EARLY_STOP_PATIENCE:
                print(f"[{class_name} seed{seed}] early stopping at epoch {epoch}")
                break

    # Save loss curve
    hist_df = pd.DataFrame(history)
    hist_csv = os.path.join(OUTPUT_LOG_DIR, f"vae_{class_name}_seed{seed}_losses.csv")
    hist_df.to_csv(hist_csv, index=False)

    # Reload best checkpoint and save qualitative panels
    best = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(best["model_state"])
    model.eval()

    with torch.no_grad():
        batch = next(iter(val_loader)).to(device)
        n_show = min(5, batch.size(0))
        x_hat, _, _ = model(batch[:n_show])
        recon_grid = torch.cat([batch[:n_show], x_hat[:n_show]], dim=0)
        save_image_grid(recon_grid, os.path.join(
            OUTPUT_FIGURE_DIR, f"vae_{class_name}_seed{seed}_reconstructions.png"), nrow=n_show)

        samples = model.sample(25, device=device)
        save_image_grid(samples, os.path.join(
            OUTPUT_FIGURE_DIR, f"vae_{class_name}_seed{seed}_samples.png"), nrow=5)

    print(f"[{class_name} seed{seed}] done. best_val_loss={best_val_loss:.4f}. "
          f"Checkpoint: {ckpt_path}")
    return ckpt_path, hist_csv


def main():
    parser = argparse.ArgumentParser(description="Train a class-specific VAE.")
    parser.add_argument("--manifest", required=True)
    parser.add_argument("--class_name", required=True)
    parser.add_argument("--seed", type=int, required=True)
    parser.add_argument("--max_epochs", type=int, default=VAE_MAX_EPOCHS)
    args = parser.parse_args()
    train_one_vae(args.manifest, args.class_name, args.seed, max_epochs=args.max_epochs)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/generate_synthetic.py
"""
Phase 8: Balance -- generate synthetic minority-class images from trained
VAEs, according to a balancing policy (C1 classical / C2 50%-cap / C3
equalized), and run basic automated quality checks.

Writes:
  data/synthetic/<class>_seed<seed>/synth_*.png
  data/manifests/synthetic_seed<seed>_<policy>.csv   (file_path, dx, source)
  outputs/predictions/synthetic_review_seed<seed>_<policy>.csv

Usage:
    python src/generate_synthetic.py --manifest data/manifests/splits_seed17.csv \
        --seed 17 --policy C3
"""
import argparse
import os
import sys

import numpy as np
import pandas as pd
import torch
from PIL import Image

sys.path.insert(0, os.path.dirname(__file__))
from config import (  # noqa: E402
    CLASS_NAMES, DATA_MANIFEST_DIR, DATA_SYNTHETIC_DIR, OUTPUT_CHECKPOINT_DIR,
    OUTPUT_PREDICTION_DIR,
)
from models_vae import ConvVAE  # noqa: E402


def compute_train_counts(manifest_path: str):
    df = pd.read_csv(manifest_path)
    train_df = df[df["partition"] == "train"]
    counts = train_df["dx"].value_counts().reindex(CLASS_NAMES).fillna(0).astype(int).to_dict()
    return counts, train_df


def targets_for_policy(train_counts: dict, policy: str):
    majority = max(train_counts.values())
    targets = {}
    for cls, n in train_counts.items():
        if policy == "C1":
            targets[cls] = n  # no VAE synthesis; classical oversampling handled separately
        elif policy == "C2":
            targets[cls] = max(n, int(round(0.5 * majority)))
        elif policy == "C3":
            targets[cls] = majority
        else:
            raise ValueError(f"Unknown policy: {policy}")
    return targets


def basic_quality_checks(img_array: np.ndarray):
    """
    Lightweight automated screen (Section 10 of the plan): flags obviously
    malformed images. This does NOT replace a manual visual review pass --
    it just catches degenerate decoder failures automatically.
    """
    flags = []
    if img_array.min() < -1e-3 or img_array.max() > 1 + 1e-3:
        flags.append("out_of_range")
    std = img_array.std()
    if std < 0.01:
        flags.append("near_blank_low_variance")
    return flags


def generate_for_class(class_name: str, n_to_generate: int, seed: int, device,
                        real_file_paths=None, mode: str = "manifold"):
    """
    mode='manifold' (default, recommended): generates each synthetic image
        by encoding TWO real training images of this class and decoding a
        random interpolation between their latent codes (plus a small noise
        nudge for variety). This keeps synthetic images anchored close to
        the real data manifold, rather than floating toward the "generic
        average" look that pure-prior sampling tends to produce -- that
        generic look is a likely source of the train/val gap seen in C3.
    mode='prior': the original behavior -- decode a random N(0,1) latent.
        Kept available for comparison / ablation.
    real_file_paths: required for mode='manifold' -- list of real training
        image paths for this class.
    """
    if n_to_generate <= 0:
        return []

    ckpt_path = os.path.join(OUTPUT_CHECKPOINT_DIR, f"vae_{class_name}_seed{seed}.pt")
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(
            f"No trained VAE checkpoint for class '{class_name}' seed {seed} at {ckpt_path}. "
            f"Run train_vae.py for this class/seed first."
        )

    model = ConvVAE().to(device)
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state["model_state"])
    model.eval()

    out_dir = os.path.join(DATA_SYNTHETIC_DIR, f"{class_name}_seed{seed}")
    os.makedirs(out_dir, exist_ok=True)

    torch.manual_seed(seed * 1000 + hash(class_name) % 1000)
    rng = np.random.RandomState(seed * 1000 + hash(class_name) % 1000)
    records = []
    latent_seed_counter = 0

    if mode == "manifold":
        if not real_file_paths or len(real_file_paths) < 2:
            raise ValueError(
                f"mode='manifold' needs at least 2 real training images for class "
                f"'{class_name}', got {len(real_file_paths) if real_file_paths else 0}. "
                f"Falling back to mode='prior' for this class is a reasonable alternative."
            )
        from datasets import build_vae_transforms
        transform = build_vae_transforms(train=False)

        # Pre-encode all real images once (cheap relative to generation loop).
        with torch.no_grad():
            real_tensors = torch.stack([
                transform(Image.open(p).convert("RGB")) for p in real_file_paths
            ]).to(device)
            mus, _ = model.encode(real_tensors)  # use mu only -- deterministic, cleaner anchor

        n_real = mus.size(0)
        batch_size = 32
        generated = 0
        while generated < n_to_generate:
            n = min(batch_size, n_to_generate - generated)
            idx_a = rng.randint(0, n_real, size=n)
            idx_b = rng.randint(0, n_real, size=n)
            alpha = torch.tensor(rng.uniform(0.3, 0.7, size=n), dtype=torch.float32, device=device).unsqueeze(1)
            z = alpha * mus[idx_a] + (1 - alpha) * mus[idx_b]
            z = z + 0.05 * torch.randn_like(z)  # small nudge so it's not just a deterministic blend
            with torch.no_grad():
                samples = model.decode(z).clamp(0.0, 1.0).cpu().numpy()

            for i in range(n):
                img_arr = samples[i]
                flags = basic_quality_checks(img_arr)
                img_uint8 = (np.clip(img_arr, 0, 1).transpose(1, 2, 0) * 255).astype(np.uint8)
                img_id = f"synth_{class_name}_seed{seed}_{latent_seed_counter:06d}"
                file_path = os.path.join(out_dir, img_id + ".png")
                Image.fromarray(img_uint8).save(file_path)
                records.append({
                    "image_id": img_id, "file_path": file_path, "dx": class_name,
                    "source": "vae_synthetic_manifold", "source_vae_checkpoint": ckpt_path,
                    "latent_seed": latent_seed_counter, "seed": seed,
                    "review_status": "rejected" if flags else "unreviewed",
                    "auto_flags": ";".join(flags),
                })
                latent_seed_counter += 1
            generated += n
        return records

    # mode == "prior": original behavior
    batch_size = 32
    generated = 0
    while generated < n_to_generate:
        n = min(batch_size, n_to_generate - generated)
        with torch.no_grad():
            samples = model.sample(n, device=device).cpu().numpy()
        for i in range(n):
            img_arr = samples[i]
            flags = basic_quality_checks(img_arr)
            img_uint8 = (np.clip(img_arr, 0, 1).transpose(1, 2, 0) * 255).astype(np.uint8)
            img_id = f"synth_{class_name}_seed{seed}_{latent_seed_counter:06d}"
            file_path = os.path.join(out_dir, img_id + ".png")
            Image.fromarray(img_uint8).save(file_path)
            records.append({
                "image_id": img_id, "file_path": file_path, "dx": class_name,
                "source": "vae_synthetic_prior", "source_vae_checkpoint": ckpt_path,
                "latent_seed": latent_seed_counter, "seed": seed,
                "review_status": "rejected" if flags else "unreviewed",
                "auto_flags": ";".join(flags),
            })
            latent_seed_counter += 1
        generated += n

    return records


def main():
    parser = argparse.ArgumentParser(description="Generate VAE synthetic images per balancing policy.")
    parser.add_argument("--manifest", required=True)
    parser.add_argument("--seed", type=int, required=True)
    parser.add_argument("--policy", choices=["C2", "C3"], required=True,
                         help="C1 uses classical oversampling, not VAE generation -- handle "
                              "that directly in train_classifier.py's sampler instead.")
    parser.add_argument("--mode", choices=["manifold", "prior"], default="manifold",
                         help="'manifold' (default): interpolate between real images' latents -- "
                              "stays closer to the real data manifold. 'prior': original random-"
                              "latent sampling, kept for comparison.")
    args = parser.parse_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    train_counts, train_df = compute_train_counts(args.manifest)
    targets = targets_for_policy(train_counts, args.policy)

    print(f"Train counts: {train_counts}")
    print(f"Targets ({args.policy}): {targets}")
    print(f"Generation mode: {args.mode}")

    all_records = []
    for cls in CLASS_NAMES:
        n_needed = max(0, targets[cls] - train_counts[cls])
        if n_needed == 0:
            continue
        print(f"Generating {n_needed} synthetic images for class '{cls}'...")
        real_paths = train_df[train_df["dx"] == cls]["file_path"].tolist() if args.mode == "manifold" else None
        recs = generate_for_class(cls, n_needed, args.seed, device,
                                   real_file_paths=real_paths, mode=args.mode)
        all_records.extend(recs)

    synth_df = pd.DataFrame(all_records)
    manifest_out = os.path.join(DATA_MANIFEST_DIR, f"synthetic_seed{args.seed}_{args.policy}.csv")
    synth_df.to_csv(manifest_out, index=False)

    review_out = os.path.join(OUTPUT_PREDICTION_DIR, f"synthetic_review_seed{args.seed}_{args.policy}.csv")
    synth_df.to_csv(review_out, index=False)

    n_flagged = (synth_df["review_status"] == "rejected").sum() if len(synth_df) else 0
    print(f"Generated {len(synth_df)} synthetic images total. "
          f"{n_flagged} auto-flagged as degenerate. Manifest: {manifest_out}")
    print("NOTE: 'unreviewed' images still need the manual visual-plausibility pass "
          "described in Section 10 of the plan before being trusted at scale.")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/train_classifier.py
"""
Phase 5 / 9: Train (or retrain) the DenseNet-121 classifier under one of the
four conditions (C0/C1/C2/C3), using the two-stage fine-tuning schedule.

Usage:
    python src/train_classifier.py --manifest data/manifests/splits_seed17.csv \
        --seed 17 --condition C0

    python src/train_classifier.py --manifest data/manifests/splits_seed17.csv \
        --seed 17 --condition C3 \
        --synthetic_manifest data/manifests/synthetic_seed17_C3.csv
"""
import argparse
import os
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, WeightedRandomSampler

sys.path.insert(0, os.path.dirname(__file__))
from config import NUM_WORKERS  # noqa: E402
from config import (  # noqa: E402
    CLASS_NAMES, CLF_BACKBONE_LR, CLF_BATCH_SIZE, CLF_EARLY_STOP_PATIENCE,
    CLF_HEAD_LR, CLF_LABEL_SMOOTHING, CLF_STAGE_A_EPOCHS, CLF_STAGE_B_MAX_EPOCHS,
    CLF_WARMUP_EPOCHS, CLF_WEIGHT_DECAY, NUM_CLASSES, OUTPUT_CHECKPOINT_DIR,
    OUTPUT_LOG_DIR, OUTPUT_PREDICTION_DIR,
)
from datasets import HAM10000ClassifierDataset  # noqa: E402
from models_densenet import (  # noqa: E402
    build_densenet121, freeze_backbone, get_param_groups, unfreeze_final_block,
)


def make_weighted_sampler(df: pd.DataFrame):
    """Classical control (C1): inverse-frequency weighted sampler, no VAE."""
    class_counts = df["dx"].value_counts()
    weights = df["dx"].map(lambda c: 1.0 / class_counts[c]).values
    return WeightedRandomSampler(weights=weights, num_samples=len(weights), replacement=True)


def run_epoch(model, loader, criterion, optimizer, device, train: bool):
    model.train() if train else model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * x.size(0)
            all_preds.append(logits.argmax(1).detach().cpu().numpy())
            all_labels.append(y.detach().cpu().numpy())
    preds = np.concatenate(all_preds)
    labels = np.concatenate(all_labels)
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    return total_loss / len(loader.dataset), macro_f1


def train_classifier(manifest_path: str, seed: int, condition: str,
                      synthetic_manifest_path: str = None,
                      max_epochs_stage_b: int = CLF_STAGE_B_MAX_EPOCHS,
                      real_only_finetune_epochs: int = 0,
                      device=None):
    """
    real_only_finetune_epochs: for conditions using synthetic data (C2/C3),
        an optional short final phase (recommended: 3-5) training on ONLY
        real images at a reduced learning rate, after the main combined-data
        training. Purpose: the combined phase lets the model benefit from
        the extra synthetic volume, but can pick up subtle synthetic-vs-real
        texture shortcuts (evidenced by a large train/val gap); this final
        real-only phase re-anchors the decision boundary to the real image
        distribution specifically. Ignored (no-op) for C0/C1.
    """
    torch.manual_seed(seed)
    np.random.seed(seed)
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    df = pd.read_csv(manifest_path)
    train_df = df[df["partition"] == "train"].copy()
    val_df = df[df["partition"] == "val"].copy()

    synthetic_df = None
    if synthetic_manifest_path:
        synthetic_df = pd.read_csv(synthetic_manifest_path)
        synthetic_df = synthetic_df[synthetic_df["review_status"] != "rejected"]

    train_ds = HAM10000ClassifierDataset(train_df, train=True, synthetic_df=synthetic_df)
    val_ds = HAM10000ClassifierDataset(val_df, train=False)

    if condition == "C1":
        combined_df = train_ds.df  # real only for C1 (no VAE synthetic)
        sampler = make_weighted_sampler(combined_df)
        train_loader = DataLoader(train_ds, batch_size=CLF_BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS)
    else:
        train_loader = DataLoader(train_ds, batch_size=CLF_BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)

    val_loader = DataLoader(val_ds, batch_size=CLF_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    model = build_densenet121(num_classes=NUM_CLASSES, pretrained=True).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=CLF_LABEL_SMOOTHING)

    run_tag = f"{condition}_seed{seed}"
    ckpt_path = os.path.join(OUTPUT_CHECKPOINT_DIR, f"densenet121_{run_tag}.pt")
    log_rows = []

    # ---- Stage A: frozen backbone, head only ----
    model = freeze_backbone(model)
    optimizer = torch.optim.AdamW(get_param_groups(model, CLF_HEAD_LR, CLF_BACKBONE_LR),
                                   weight_decay=CLF_WEIGHT_DECAY)
    for epoch in range(1, CLF_STAGE_A_EPOCHS + 1):
        train_loss, train_f1 = run_epoch(model, train_loader, criterion, optimizer, device, train=True)
        val_loss, val_f1 = run_epoch(model, val_loader, criterion, optimizer, device, train=False)
        print(f"[{run_tag}] StageA epoch {epoch}/{CLF_STAGE_A_EPOCHS} "
              f"train_loss={train_loss:.4f} train_f1={train_f1:.4f} "
              f"val_loss={val_loss:.4f} val_f1={val_f1:.4f}")
        log_rows.append({"stage": "A", "epoch": epoch, "train_loss": train_loss,
                          "train_macro_f1": train_f1, "val_loss": val_loss, "val_macro_f1": val_f1})

    # ---- Stage B: unfreeze final block, discriminative LR fine-tuning ----
    model = unfreeze_final_block(model)
    optimizer = torch.optim.AdamW(get_param_groups(model, CLF_HEAD_LR, CLF_BACKBONE_LR),
                                   weight_decay=CLF_WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs_stage_b)

    best_val_f1 = -1.0
    epochs_no_improve = 0

    for epoch in range(1, max_epochs_stage_b + 1):
        train_loss, train_f1 = run_epoch(model, train_loader, criterion, optimizer, device, train=True)
        val_loss, val_f1 = run_epoch(model, val_loader, criterion, optimizer, device, train=False)
        scheduler.step()

        print(f"[{run_tag}] StageB epoch {epoch}/{max_epochs_stage_b} "
              f"train_loss={train_loss:.4f} train_f1={train_f1:.4f} "
              f"val_loss={val_loss:.4f} val_f1={val_f1:.4f}")
        log_rows.append({"stage": "B", "epoch": epoch, "train_loss": train_loss,
                          "train_macro_f1": train_f1, "val_loss": val_loss, "val_macro_f1": val_f1})

        if val_f1 > best_val_f1 + 1e-5:
            best_val_f1 = val_f1
            epochs_no_improve = 0
            torch.save({"model_state": model.state_dict(), "epoch": epoch,
                        "condition": condition, "seed": seed, "val_macro_f1": val_f1}, ckpt_path)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= CLF_EARLY_STOP_PATIENCE:
                print(f"[{run_tag}] early stopping at epoch {epoch} (best val_f1={best_val_f1:.4f})")
                break

    log_df = pd.DataFrame(log_rows)
    log_csv = os.path.join(OUTPUT_LOG_DIR, f"densenet121_{run_tag}_log.csv")
    log_df.to_csv(log_csv, index=False)

    # ---- Stage C (optional): short real-only fine-tune to correct any
    # synthetic-vs-real shortcut the model picked up during combined training ----
    if real_only_finetune_epochs > 0 and condition in ("C2", "C3"):
        print(f"[{run_tag}] Starting Stage C: real-only fine-tune "
              f"({real_only_finetune_epochs} epochs, reduced LR)")

        # Reload best combined-training checkpoint as the starting point.
        best_state = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(best_state["model_state"])

        real_only_ds = HAM10000ClassifierDataset(train_df, train=True, synthetic_df=None)
        real_only_loader = DataLoader(real_only_ds, batch_size=CLF_BATCH_SIZE, shuffle=True,
                                       num_workers=NUM_WORKERS)

        stage_c_optimizer = torch.optim.AdamW(
            get_param_groups(model, CLF_HEAD_LR * 0.1, CLF_BACKBONE_LR * 0.1),
            weight_decay=CLF_WEIGHT_DECAY,
        )

        for epoch in range(1, real_only_finetune_epochs + 1):
            train_loss, train_f1 = run_epoch(model, real_only_loader, criterion, stage_c_optimizer, device, train=True)
            val_loss, val_f1 = run_epoch(model, val_loader, criterion, stage_c_optimizer, device, train=False)

            print(f"[{run_tag}] StageC epoch {epoch}/{real_only_finetune_epochs} "
                  f"train_loss={train_loss:.4f} train_f1={train_f1:.4f} "
                  f"val_loss={val_loss:.4f} val_f1={val_f1:.4f}")
            log_rows.append({"stage": "C", "epoch": epoch, "train_loss": train_loss,
                              "train_macro_f1": train_f1, "val_loss": val_loss, "val_macro_f1": val_f1})

            if val_f1 > best_val_f1 + 1e-5:
                best_val_f1 = val_f1
                torch.save({"model_state": model.state_dict(), "epoch": epoch,
                            "condition": condition, "seed": seed, "val_macro_f1": val_f1,
                            "stage": "C"}, ckpt_path)
                print(f"[{run_tag}] StageC improved val_f1 to {best_val_f1:.4f} -- checkpoint updated.")

        log_df = pd.DataFrame(log_rows)
        log_df.to_csv(log_csv, index=False)

    print(f"[{run_tag}] done. best_val_macro_f1={best_val_f1:.4f}. Checkpoint: {ckpt_path}")
    return ckpt_path, log_csv


def main():
    parser = argparse.ArgumentParser(description="Train/retrain DenseNet-121 under one condition.")
    parser.add_argument("--manifest", required=True)
    parser.add_argument("--seed", type=int, required=True)
    parser.add_argument("--condition", choices=["C0", "C1", "C2", "C3"], required=True)
    parser.add_argument("--synthetic_manifest", default=None,
                         help="Required for C2/C3, path to generate_synthetic.py output CSV.")
    parser.add_argument("--max_epochs_stage_b", type=int, default=CLF_STAGE_B_MAX_EPOCHS)
    parser.add_argument("--real_only_finetune_epochs", type=int, default=0,
                         help="Optional short real-only polish phase after combined training "
                              "(C2/C3 only). Recommended: 3-5.")
    args = parser.parse_args()

    if args.condition in ("C2", "C3") and not args.synthetic_manifest:
        raise ValueError(f"Condition {args.condition} requires --synthetic_manifest "
                          f"(output of generate_synthetic.py).")

    train_classifier(args.manifest, args.seed, args.condition,
                      synthetic_manifest_path=args.synthetic_manifest,
                      max_epochs_stage_b=args.max_epochs_stage_b,
                      real_only_finetune_epochs=args.real_only_finetune_epochs)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/evaluate.py
"""
Phase 10 / 11: Evaluate trained checkpoints on the frozen real-only test
set, compute the full metric suite, and (optionally) a paired bootstrap
comparison between two conditions (e.g. C0 vs C3).

Usage:
    python src/evaluate.py --manifest data/manifests/splits_seed17.csv \
        --checkpoint outputs/checkpoints/densenet121_C0_seed17.pt \
        --condition C0 --seed 17

    python src/evaluate.py --compare \
        --pred_a outputs/predictions/preds_C0_seed17.csv \
        --pred_b outputs/predictions/preds_C3_seed17.csv
"""
import argparse
import os
import sys

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    average_precision_score, balanced_accuracy_score, confusion_matrix,
    f1_score, precision_recall_fscore_support, roc_auc_score,
)
from torch.utils.data import DataLoader

sys.path.insert(0, os.path.dirname(__file__))
from config import CLASS_NAMES, NUM_CLASSES, NUM_WORKERS, OUTPUT_FIGURE_DIR, OUTPUT_PREDICTION_DIR  # noqa: E402
from datasets import HAM10000ClassifierDataset  # noqa: E402
from models_densenet import build_densenet121  # noqa: E402


def expected_calibration_error(probs: np.ndarray, labels: np.ndarray, n_bins: int = 15):
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    accuracies = (predictions == labels).astype(float)

    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        mask = (confidences > lo) & (confidences <= hi) if i > 0 else (confidences >= lo) & (confidences <= hi)
        if mask.sum() == 0:
            continue
        bin_acc = accuracies[mask].mean()
        bin_conf = confidences[mask].mean()
        ece += (mask.sum() / len(confidences)) * abs(bin_acc - bin_conf)
    return ece


def compute_metrics(labels: np.ndarray, preds: np.ndarray, probs: np.ndarray):
    accuracy = (labels == preds).mean()
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    weighted_f1 = f1_score(labels, preds, average="weighted", zero_division=0)
    bal_acc = balanced_accuracy_score(labels, preds)
    precision, recall, f1_per_class, support = precision_recall_fscore_support(
        labels, preds, labels=list(range(NUM_CLASSES)), zero_division=0
    )

    try:
        macro_auroc = roc_auc_score(labels, probs, multi_class="ovr", average="macro")
    except ValueError:
        macro_auroc = float("nan")

    auprc_per_class = []
    for c in range(NUM_CLASSES):
        y_true_c = (labels == c).astype(int)
        try:
            auprc_per_class.append(average_precision_score(y_true_c, probs[:, c]))
        except ValueError:
            auprc_per_class.append(float("nan"))

    ece = expected_calibration_error(probs, labels)
    cm = confusion_matrix(labels, preds, labels=list(range(NUM_CLASSES)))

    per_class_df = pd.DataFrame({
        "class": CLASS_NAMES, "precision": precision, "recall": recall,
        "f1": f1_per_class, "support": support, "auprc": auprc_per_class,
    })

    summary = {
        "accuracy": accuracy, "macro_f1": macro_f1, "weighted_f1": weighted_f1,
        "balanced_accuracy": bal_acc, "macro_auroc": macro_auroc, "ece": ece,
    }
    return summary, per_class_df, cm


def predict_on_test(checkpoint_path: str, manifest_path: str, device=None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    df = pd.read_csv(manifest_path)
    test_df = df[df["partition"] == "test"].copy()

    ds = HAM10000ClassifierDataset(test_df, train=False)
    loader = DataLoader(ds, batch_size=32, shuffle=False, num_workers=NUM_WORKERS)

    model = build_densenet121(num_classes=NUM_CLASSES, pretrained=False).to(device)
    state = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state["model_state"])
    model.eval()

    all_probs, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            logits = model(x)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            all_probs.append(probs)
            all_labels.append(y.numpy())

    probs = np.concatenate(all_probs)
    labels = np.concatenate(all_labels)
    preds = probs.argmax(axis=1)

    # Attach group id for paired bootstrap later.
    group_ids = test_df["lesion_id"].values if "lesion_id" in test_df.columns else np.arange(len(labels))

    return probs, preds, labels, group_ids


def paired_bootstrap_ci(labels, preds_a, preds_b, group_ids, n_boot=2000, seed=0, metric="macro_f1"):
    """95% CI for macro-F1 difference (B - A), resampling by lesion group."""
    rng = np.random.RandomState(seed)
    unique_groups = np.unique(group_ids)
    diffs = []
    for _ in range(n_boot):
        sampled_groups = rng.choice(unique_groups, size=len(unique_groups), replace=True)
        mask_idx = np.concatenate([np.where(group_ids == g)[0] for g in sampled_groups])
        f1_a = f1_score(labels[mask_idx], preds_a[mask_idx], average="macro", zero_division=0)
        f1_b = f1_score(labels[mask_idx], preds_b[mask_idx], average="macro", zero_division=0)
        diffs.append(f1_b - f1_a)
    diffs = np.array(diffs)
    return {
        "mean_diff": diffs.mean(),
        "ci_lower": np.percentile(diffs, 2.5),
        "ci_upper": np.percentile(diffs, 97.5),
    }


def plot_confusion_matrix(cm, out_path, title):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(NUM_CLASSES))
    ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES, rotation=45)
    ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8)
    fig.colorbar(im)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def main():
    parser = argparse.ArgumentParser(description="Evaluate a classifier checkpoint or compare two conditions.")
    parser.add_argument("--manifest")
    parser.add_argument("--checkpoint")
    parser.add_argument("--condition")
    parser.add_argument("--seed", type=int)
    parser.add_argument("--compare", action="store_true")
    parser.add_argument("--pred_a")
    parser.add_argument("--pred_b")
    args = parser.parse_args()

    if args.compare:
        df_a = pd.read_csv(args.pred_a)
        df_b = pd.read_csv(args.pred_b)
        labels = df_a["label"].values
        result = paired_bootstrap_ci(
            labels, df_a["pred"].values, df_b["pred"].values, df_a["group_id"].values
        )
        print("Paired bootstrap macro-F1 difference (B - A):")
        print(result)
        return

    probs, preds, labels, group_ids = predict_on_test(args.checkpoint, args.manifest)
    summary, per_class_df, cm = compute_metrics(labels, preds, probs)

    run_tag = f"{args.condition}_seed{args.seed}"
    print(f"=== Results: {run_tag} ===")
    for k, v in summary.items():
        print(f"  {k}: {v:.4f}")
    print(per_class_df.to_string(index=False))

    pred_df = pd.DataFrame({
        "label": labels, "pred": preds, "group_id": group_ids,
    })
    for c in range(NUM_CLASSES):
        pred_df[f"prob_{CLASS_NAMES[c]}"] = probs[:, c]
    pred_csv = os.path.join(OUTPUT_PREDICTION_DIR, f"preds_{run_tag}.csv")
    pred_df.to_csv(pred_csv, index=False)

    per_class_csv = os.path.join(OUTPUT_PREDICTION_DIR, f"per_class_metrics_{run_tag}.csv")
    per_class_df.to_csv(per_class_csv, index=False)

    summary_csv = os.path.join(OUTPUT_PREDICTION_DIR, f"summary_metrics_{run_tag}.csv")
    pd.DataFrame([summary]).to_csv(summary_csv, index=False)

    cm_path = os.path.join(OUTPUT_FIGURE_DIR, f"confusion_matrix_{run_tag}.png")
    plot_confusion_matrix(cm, cm_path, f"Confusion matrix: {run_tag}")

    print(f"Saved predictions -> {pred_csv}")
    print(f"Saved per-class metrics -> {per_class_csv}")
    print(f"Saved summary metrics -> {summary_csv}")
    print(f"Saved confusion matrix plot -> {cm_path}")


if __name__ == "__main__":
    main()


## Locate the attached dataset

Kaggle sometimes nests the CSV/image folders differently between dataset
versions -- this cell finds them robustly instead of hard-coding a path.


In [ ]:
import glob

candidates = glob.glob('/kaggle/input/**/HAM10000_metadata.csv', recursive=True)
assert candidates, "Could not find HAM10000_metadata.csv -- did you attach the dataset via 'Add Input'?"
METADATA_CSV = candidates[0]
DATASET_ROOT = os.path.dirname(METADATA_CSV)

IMAGE_DIRS = [d for d in glob.glob(os.path.join(DATASET_ROOT, '*')) if os.path.isdir(d) and 'image' in d.lower()]
if not IMAGE_DIRS:
    # some dataset versions nest one level deeper
    IMAGE_DIRS = [d for d in glob.glob(os.path.join(DATASET_ROOT, '*', '*')) if os.path.isdir(d) and 'image' in d.lower()]

print('Metadata CSV:', METADATA_CSV)
print('Image directories found:')
for d in IMAGE_DIRS:
    print(' -', d)
assert IMAGE_DIRS, "Could not find image folders -- check the dataset structure with `glob.glob('/kaggle/input/**', recursive=True)`."


---
# SESSION 1: Audit + Split (fast -- CPU is fine, no GPU needed)


In [ ]:
import sys
sys.path.insert(0, 'src')
import audit_data

audit_df, group_col = audit_data.run_audit(METADATA_CSV, IMAGE_DIRS, compute_hashes=True)


In [ ]:
import make_group_splits
import pandas as pd
from config import SEEDS

valid_df = pd.read_csv('data/processed/audit.csv')
valid_df = valid_df[valid_df['file_exists'] & valid_df['label_valid']].copy()
if valid_df['is_exact_duplicate_file'].any():
    valid_df = valid_df.sort_values('image_id').drop_duplicates(subset='md5', keep='first')
    print(f'Dropped exact-duplicate files. Remaining rows: {len(valid_df)}')

for seed in SEEDS:
    split_df = make_group_splits.grouped_stratified_split(valid_df, group_col='lesion_id', label_col='dx', seed=seed)
    out_path = f'data/manifests/splits_seed{seed}.csv'
    split_df.to_csv(out_path, index=False)
    print(f'=== seed {seed} -> {out_path} ===')
    make_group_splits.report_split(split_df, 'dx')


**Checkpoint:** click *Save Version* now so `data/processed/` and `data/manifests/` persist as this notebook's output for the next session.

---
# SESSION 2: Baseline classifier (condition C0)

Turn the GPU accelerator ON for this session (Settings -> Accelerator -> GPU).
If continuing from a previous session, re-run the "write out project source
files" and "locate dataset" cells above first, or attach your Session-1
output as an additional input dataset.


In [ ]:
import sys
sys.path.insert(0, 'src')
import train_classifier

SEED = 17  # repeat this whole session block with seed=29 and seed=41 when you have quota

train_classifier.train_classifier(
    manifest_path=f'data/manifests/splits_seed{SEED}.csv',
    seed=SEED,
    condition='C0',
)


In [ ]:
import evaluate

probs, preds, labels, group_ids = evaluate.predict_on_test(
    f'outputs/checkpoints/densenet121_C0_seed{SEED}.pt',
    f'data/manifests/splits_seed{SEED}.csv',
)
summary, per_class_df, cm = evaluate.compute_metrics(labels, preds, probs)
print('=== C0 baseline results (seed', SEED, ') ===')
print(summary)
print(per_class_df)

import pandas as pd
pred_df = pd.DataFrame({'label': labels, 'pred': preds, 'group_id': group_ids})
from config import CLASS_NAMES
for c, name in enumerate(CLASS_NAMES):
    pred_df[f'prob_{name}'] = probs[:, c]
pred_df.to_csv(f'outputs/predictions/preds_C0_seed{SEED}.csv', index=False)
per_class_df.to_csv(f'outputs/predictions/per_class_metrics_C0_seed{SEED}.csv', index=False)
pd.DataFrame([summary]).to_csv(f'outputs/predictions/summary_metrics_C0_seed{SEED}.csv', index=False)
evaluate.plot_confusion_matrix(cm, f'outputs/figures/confusion_matrix_C0_seed{SEED}.png', f'C0 seed{SEED}')
print('Saved C0 predictions and metrics.')


**Checkpoint:** Save Version. This is your first real baseline result.

---
# SESSION 3: Train the minority-class VAEs

This is the slowest phase (up to 150 epochs each, with early stopping).
Watch the printed val_loss -- if it never improves, or the sample panels
in `outputs/figures/vae_<class>_seed{SEED}_samples.png` look like noise or
solid color, that class likely has too few images (expect this risk
especially for `df` and `vasc`).


In [ ]:
import train_vae

for cls in ['mel', 'bkl', 'bcc', 'akiec', 'vasc', 'df']:
    print(f'=== Training VAE for class: {cls} (seed {SEED}) ===')
    train_vae.train_one_vae(f'data/manifests/splits_seed{SEED}.csv', cls, SEED)


**Checkpoint:** Save Version. Inspect each `vae_<class>_seed17_samples.png` and `_reconstructions.png` in the Output tab before continuing.

---
# SESSION 4: Generate synthetic images (condition C3: fully equalized) + retrain + evaluate

**Update:** generation now defaults to `mode='manifold'` -- each synthetic image is decoded from an interpolation between two REAL images' latent codes, rather than a fully random point. This keeps synthetic images closer to the real data distribution (less of the generic 'averaged' look pure random sampling tends to produce). The classifier retraining step also now runs an optional short **real-only fine-tune phase** afterward, to correct for any synthetic-vs-real shortcut the model might otherwise learn.


In [ ]:
import generate_synthetic

device = 'cuda' if torch.cuda.is_available() else 'cpu'
train_counts, train_df = generate_synthetic.compute_train_counts(f'data/manifests/splits_seed{SEED}.csv')
targets = generate_synthetic.targets_for_policy(train_counts, 'C3')
print('Train counts:', train_counts)
print('C3 targets:', targets)

all_records = []
from config import CLASS_NAMES
for cls in CLASS_NAMES:
    n_needed = max(0, targets[cls] - train_counts[cls])
    if n_needed == 0:
        continue
    print(f'Generating {n_needed} synthetic images for class \'{cls}\'...')
    real_paths_this_class = train_df[train_df['dx'] == cls]['file_path'].tolist()
    recs = generate_synthetic.generate_for_class(
        cls, n_needed, SEED, device, real_file_paths=real_paths_this_class, mode='manifold'
    )
    all_records.extend(recs)

import pandas as pd
synth_df = pd.DataFrame(all_records)
synth_df.to_csv(f'data/manifests/synthetic_seed{SEED}_C3.csv', index=False)
synth_df.to_csv(f'outputs/predictions/synthetic_review_seed{SEED}_C3.csv', index=False)
n_flagged = (synth_df['review_status'] == 'rejected').sum() if len(synth_df) else 0
print(f'Generated {len(synth_df)} synthetic images. {n_flagged} auto-flagged as degenerate.')
print('IMPORTANT: manually spot-check outputs/predictions/synthetic_review_*.csv and the')
print('sample image files before trusting these at scale (Section 10 of the plan).')


In [ ]:
train_classifier.train_classifier(
    manifest_path=f'data/manifests/splits_seed{SEED}.csv',
    seed=SEED,
    condition='C3',
    synthetic_manifest_path=f'data/manifests/synthetic_seed{SEED}_C3.csv',
    real_only_finetune_epochs=3,  # short real-only polish phase, see notes above
)


In [ ]:
probs3, preds3, labels3, group_ids3 = evaluate.predict_on_test(
    f'outputs/checkpoints/densenet121_C3_seed{SEED}.pt',
    f'data/manifests/splits_seed{SEED}.csv',
)
summary3, per_class_df3, cm3 = evaluate.compute_metrics(labels3, preds3, probs3)
print('=== C3 VAE-equalized results (seed', SEED, ') ===')
print(summary3)
print(per_class_df3)

pred_df3 = pd.DataFrame({'label': labels3, 'pred': preds3, 'group_id': group_ids3})
for c, name in enumerate(CLASS_NAMES):
    pred_df3[f'prob_{name}'] = probs3[:, c]
pred_df3.to_csv(f'outputs/predictions/preds_C3_seed{SEED}.csv', index=False)
per_class_df3.to_csv(f'outputs/predictions/per_class_metrics_C3_seed{SEED}.csv', index=False)
pd.DataFrame([summary3]).to_csv(f'outputs/predictions/summary_metrics_C3_seed{SEED}.csv', index=False)
evaluate.plot_confusion_matrix(cm3, f'outputs/figures/confusion_matrix_C3_seed{SEED}.png', f'C3 seed{SEED}')
print('Saved C3 predictions and metrics.')


## Compare C0 vs C3 -- the primary result this project is designed to answer

In [ ]:
import numpy as np
result = evaluate.paired_bootstrap_ci(labels, preds, preds3, group_ids, n_boot=2000, seed=0)
print('Paired bootstrap macro-F1 difference (C3 - C0):')
print(result)
print()
print(f"C0 macro-F1: {summary['macro_f1']:.4f}   C3 macro-F1: {summary3['macro_f1']:.4f}")
print(f"C0 balanced accuracy: {summary['balanced_accuracy']:.4f}   C3 balanced accuracy: {summary3['balanced_accuracy']:.4f}")
print()
print('Per-class recall comparison:')
comparison = per_class_df[['class','recall']].merge(per_class_df3[['class','recall']], on='class', suffixes=('_C0','_C3'))
print(comparison)


---
# Diagnostic: are synthetic images near-duplicates of each other?

If C3 shows high train accuracy but weak validation performance, this
checks a likely cause: whether the VAE-generated images for a class are
much less diverse (more redundant/similar to each other) than the real
images for that class. If so, the classifier may be memorizing a narrow
synthetic pattern rather than learning generalizable features.

**Note on interpreting the numbers:** because pixel values are
non-negative, absolute cosine similarity values will look high for
everything (often 0.8-0.99) -- that's a property of the metric, not a
sign images are identical. What matters is the **gap** between the real
bar and the synthetic bar for each class, not the absolute height.


In [ ]:
%%writefile src/diagnose_diversity.py
"""
Diagnostic: quantifies whether VAE-generated synthetic images are
near-duplicates of each other (low diversity) compared to real images of
the same class. This directly tests the "classifier is memorizing a
narrow synthetic pattern rather than learning generalizable features"
hypothesis raised when train accuracy is high but val accuracy lags.

Method: resize images to a small grayscale thumbnail, flatten to a vector,
L2-normalize, and compute mean pairwise cosine similarity within a class's
real images, within its synthetic images, and across real-vs-synthetic.
Higher similarity = less diversity (more redundant / duplicate-like).
This is intentionally simple and dependency-light (no extra packages
beyond what's already used) -- it's a diagnostic signal, not a
publication-grade perceptual metric.

Usage:
    python src/diagnose_diversity.py --manifest data/manifests/splits_seed17.csv \
        --synthetic_manifest data/manifests/synthetic_seed17_C3.csv --seed 17
"""
import argparse
import os
import sys

import numpy as np
import pandas as pd
from PIL import Image

sys.path.insert(0, os.path.dirname(__file__))
from config import CLASS_NAMES, OUTPUT_FIGURE_DIR, OUTPUT_PREDICTION_DIR  # noqa: E402

THUMB_SIZE = 32  # small on purpose -- we want coarse structural similarity, not pixel-exact


def load_feature_vectors(file_paths, max_n=40, seed=0):
    rng = np.random.RandomState(seed)
    paths = list(file_paths)
    if len(paths) > max_n:
        paths = list(rng.choice(paths, size=max_n, replace=False))

    vecs = []
    for p in paths:
        try:
            img = Image.open(p).convert("L").resize((THUMB_SIZE, THUMB_SIZE))
            arr = np.asarray(img, dtype=np.float32).flatten()
            norm = np.linalg.norm(arr)
            if norm > 0:
                arr = arr / norm
            vecs.append(arr)
        except Exception as e:
            print(f"  (skipped unreadable file {p}: {e})")
    if not vecs:
        return np.zeros((0, THUMB_SIZE * THUMB_SIZE))
    return np.stack(vecs)


def mean_pairwise_cosine_sim(vecs_a, vecs_b=None):
    """If vecs_b is None, computes within-set similarity (excluding self-pairs)."""
    if len(vecs_a) < 2 and vecs_b is None:
        return float("nan")
    if vecs_b is None:
        sim_matrix = vecs_a @ vecs_a.T
        n = len(vecs_a)
        mask = ~np.eye(n, dtype=bool)
        return sim_matrix[mask].mean()
    if len(vecs_a) == 0 or len(vecs_b) == 0:
        return float("nan")
    sim_matrix = vecs_a @ vecs_b.T
    return sim_matrix.mean()


def run_diversity_diagnostic(manifest_path: str, synthetic_manifest_path: str, seed: int):
    df = pd.read_csv(manifest_path)
    train_df = df[df["partition"] == "train"]
    synth_df = pd.read_csv(synthetic_manifest_path)
    synth_df = synth_df[synth_df["review_status"] != "rejected"]

    rows = []
    for cls in CLASS_NAMES:
        real_paths = train_df[train_df["dx"] == cls]["file_path"].tolist()
        synth_paths = synth_df[synth_df["dx"] == cls]["file_path"].tolist()

        real_vecs = load_feature_vectors(real_paths, seed=seed)
        synth_vecs = load_feature_vectors(synth_paths, seed=seed)

        real_internal_sim = mean_pairwise_cosine_sim(real_vecs)
        synth_internal_sim = mean_pairwise_cosine_sim(synth_vecs) if len(synth_vecs) >= 2 else float("nan")
        cross_sim = mean_pairwise_cosine_sim(real_vecs, synth_vecs) if len(synth_vecs) > 0 else float("nan")

        rows.append({
            "class": cls,
            "n_real": len(real_vecs),
            "n_synthetic": len(synth_vecs),
            "real_internal_similarity": real_internal_sim,
            "synthetic_internal_similarity": synth_internal_sim,
            "real_vs_synthetic_similarity": cross_sim,
            "synthetic_more_redundant_than_real": (
                (synth_internal_sim - real_internal_sim) if not np.isnan(synth_internal_sim) else np.nan
            ),
        })

    result_df = pd.DataFrame(rows)
    print("=" * 100)
    print("SYNTHETIC IMAGE DIVERSITY DIAGNOSTIC")
    print("Higher 'similarity' = LESS diverse (images more redundant/near-duplicate-like).")
    print("=" * 100)
    print(result_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print("-" * 100)

    flagged = result_df[result_df["synthetic_more_redundant_than_real"] > 0.05]
    if len(flagged) > 0:
        print("Classes where synthetic images are NOTABLY less diverse than real images "
              "(gap > 0.05 -- a plausible contributor to classifier overfitting on these classes):")
        print(flagged[["class", "synthetic_more_redundant_than_real"]].to_string(index=False))
    else:
        print("No class shows a large diversity gap by this metric -- low synthetic diversity "
              "is likely NOT the main explanation here; consider other causes (e.g. sheer volume "
              "of near-identical *counts* even if not near-identical *images*, or distribution "
              "shift between VAE output style and real images).")

    csv_path = os.path.join(OUTPUT_PREDICTION_DIR, f"diversity_diagnostic_seed{seed}.csv")
    result_df.to_csv(csv_path, index=False)
    print(f"\nSaved full table to: {csv_path}")

    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt

        plot_df = result_df.dropna(subset=["synthetic_internal_similarity"])
        if len(plot_df) > 0:
            x = np.arange(len(plot_df))
            width = 0.35
            fig, ax = plt.subplots(figsize=(9, 5))
            ax.bar(x - width / 2, plot_df["real_internal_similarity"], width, label="Real images (within-class)")
            ax.bar(x + width / 2, plot_df["synthetic_internal_similarity"], width, label="Synthetic images (within-class)")
            ax.set_xticks(x)
            ax.set_xticklabels(plot_df["class"])
            ax.set_ylabel("Mean pairwise cosine similarity\n(higher = less diverse)")
            ax.set_title(f"Real vs. synthetic image diversity per class (seed {seed})")
            ax.legend()
            plt.tight_layout()
            fig_path = os.path.join(OUTPUT_FIGURE_DIR, f"diversity_comparison_seed{seed}.png")
            plt.savefig(fig_path, dpi=150)
            plt.close()
            print(f"Saved comparison plot to: {fig_path}")
    except ImportError:
        pass

    return result_df


def main():
    parser = argparse.ArgumentParser(description="Diagnose synthetic vs real image diversity per class.")
    parser.add_argument("--manifest", required=True)
    parser.add_argument("--synthetic_manifest", required=True)
    parser.add_argument("--seed", type=int, required=True)
    args = parser.parse_args()
    run_diversity_diagnostic(args.manifest, args.synthetic_manifest, args.seed)


if __name__ == "__main__":
    main()


In [ ]:
import diagnose_diversity

diversity_df = diagnose_diversity.run_diversity_diagnostic(
    manifest_path=f'data/manifests/splits_seed{SEED}.csv',
    synthetic_manifest_path=f'data/manifests/synthetic_seed{SEED}_C3.csv',
    seed=SEED,
)


---
# SESSION 5: Condition C1 -- classical oversampling (no VAE)

This is the missing control that tells you whether ANY balancing method
helps here, or whether balancing itself isn't the bottleneck for this
dataset/classifier. No synthetic images needed -- C1 uses a weighted
sampler that shows real minority-class images more often per epoch.


In [ ]:
train_classifier.train_classifier(
    manifest_path=f'data/manifests/splits_seed{SEED}.csv',
    seed=SEED,
    condition='C1',
)


In [ ]:
probs1, preds1, labels1, group_ids1 = evaluate.predict_on_test(
    f'outputs/checkpoints/densenet121_C1_seed{SEED}.pt',
    f'data/manifests/splits_seed{SEED}.csv',
)
summary1, per_class_df1, cm1 = evaluate.compute_metrics(labels1, preds1, probs1)
print('=== C1 classical-oversampling results (seed', SEED, ') ===')
print(summary1)
print(per_class_df1)

pred_df1 = pd.DataFrame({'label': labels1, 'pred': preds1, 'group_id': group_ids1})
for c, name in enumerate(CLASS_NAMES):
    pred_df1[f'prob_{name}'] = probs1[:, c]
pred_df1.to_csv(f'outputs/predictions/preds_C1_seed{SEED}.csv', index=False)
per_class_df1.to_csv(f'outputs/predictions/per_class_metrics_C1_seed{SEED}.csv', index=False)
pd.DataFrame([summary1]).to_csv(f'outputs/predictions/summary_metrics_C1_seed{SEED}.csv', index=False)
evaluate.plot_confusion_matrix(cm1, f'outputs/figures/confusion_matrix_C1_seed{SEED}.png', f'C1 seed{SEED}')
print('Saved C1 predictions and metrics.')


**Checkpoint:** Save Version now.

---
# SESSION 6: Condition C2 -- VAE-balanced to 50% of majority

Lighter-touch VAE balancing than C3 (which equalizes fully). Fewer
synthetic images means less opportunity for any synthetic-vs-real
shortcut to dominate training -- worth testing directly rather than
assuming C3's result generalizes to "VAE balancing never helps."


In [ ]:
train_counts, train_df = generate_synthetic.compute_train_counts(f'data/manifests/splits_seed{SEED}.csv')
targets_c2 = generate_synthetic.targets_for_policy(train_counts, 'C2')
print('Train counts:', train_counts)
print('C2 targets:', targets_c2)

all_records_c2 = []
for cls in CLASS_NAMES:
    n_needed = max(0, targets_c2[cls] - train_counts[cls])
    if n_needed == 0:
        continue
    print(f"Generating {n_needed} synthetic images for class '{cls}'...")
    real_paths_this_class = train_df[train_df['dx'] == cls]['file_path'].tolist()
    recs = generate_synthetic.generate_for_class(
        cls, n_needed, SEED, device, real_file_paths=real_paths_this_class, mode='manifold'
    )
    all_records_c2.extend(recs)

synth_df_c2 = pd.DataFrame(all_records_c2)
synth_df_c2.to_csv(f'data/manifests/synthetic_seed{SEED}_C2.csv', index=False)
synth_df_c2.to_csv(f'outputs/predictions/synthetic_review_seed{SEED}_C2.csv', index=False)
n_flagged_c2 = (synth_df_c2['review_status'] == 'rejected').sum() if len(synth_df_c2) else 0
print(f'Generated {len(synth_df_c2)} synthetic images. {n_flagged_c2} auto-flagged as degenerate.')


In [ ]:
train_classifier.train_classifier(
    manifest_path=f'data/manifests/splits_seed{SEED}.csv',
    seed=SEED,
    condition='C2',
    synthetic_manifest_path=f'data/manifests/synthetic_seed{SEED}_C2.csv',
    real_only_finetune_epochs=3,
)


In [ ]:
probs2, preds2, labels2, group_ids2 = evaluate.predict_on_test(
    f'outputs/checkpoints/densenet121_C2_seed{SEED}.pt',
    f'data/manifests/splits_seed{SEED}.csv',
)
summary2, per_class_df2, cm2 = evaluate.compute_metrics(labels2, preds2, probs2)
print('=== C2 VAE-50%-cap results (seed', SEED, ') ===')
print(summary2)
print(per_class_df2)

pred_df2 = pd.DataFrame({'label': labels2, 'pred': preds2, 'group_id': group_ids2})
for c, name in enumerate(CLASS_NAMES):
    pred_df2[f'prob_{name}'] = probs2[:, c]
pred_df2.to_csv(f'outputs/predictions/preds_C2_seed{SEED}.csv', index=False)
per_class_df2.to_csv(f'outputs/predictions/per_class_metrics_C2_seed{SEED}.csv', index=False)
pd.DataFrame([summary2]).to_csv(f'outputs/predictions/summary_metrics_C2_seed{SEED}.csv', index=False)
evaluate.plot_confusion_matrix(cm2, f'outputs/figures/confusion_matrix_C2_seed{SEED}.png', f'C2 seed{SEED}')
print('Saved C2 predictions and metrics.')


**Checkpoint:** Save Version now.

---
# FINAL COMPARISON: all four conditions side by side

This is the complete answer to the project's research question at this
seed. Macro-F1, balanced accuracy, and per-class recall for C0/C1/C2/C3,
plus paired bootstrap significance vs. the C0 baseline for each.


In [ ]:
all_summaries = {'C0': summary, 'C1': summary1, 'C2': summary2, 'C3': summary3}
summary_table = pd.DataFrame(all_summaries).T
summary_table.index.name = 'condition'
print('=== Summary metrics, all conditions (seed', SEED, ') ===')
print(summary_table)

all_per_class = {'C0': per_class_df, 'C1': per_class_df1, 'C2': per_class_df2, 'C3': per_class_df3}
recall_table = pd.DataFrame({name: df.set_index('class')['recall'] for name, df in all_per_class.items()})
print()
print('=== Per-class recall, all conditions ===')
print(recall_table)

print()
print('=== Paired bootstrap macro-F1 vs. C0 baseline ===')
for name, preds_x, labels_x in [('C1', preds1, labels1), ('C2', preds2, labels2), ('C3', preds3, labels3)]:
    res = evaluate.paired_bootstrap_ci(labels, preds, preds_x, group_ids, n_boot=2000, seed=0)
    sig = 'SIGNIFICANT' if (res['ci_lower'] > 0 or res['ci_upper'] < 0) else 'not significant (CI includes 0)'
    print(f"{name} - C0: mean_diff={res['mean_diff']:.4f}  95% CI=[{res['ci_lower']:.4f}, {res['ci_upper']:.4f}]  -> {sig}")

summary_table.to_csv(f'outputs/predictions/FINAL_summary_all_conditions_seed{SEED}.csv')
recall_table.to_csv(f'outputs/predictions/FINAL_recall_all_conditions_seed{SEED}.csv')
print()
print('Saved final comparison tables to outputs/predictions/FINAL_*.csv')


## How to read this table

- If none of C1/C2/C3 show a 95% CI excluding zero vs. C0, the honest,
  defensible conclusion is: **at this seed, no balancing strategy tested
  (classical oversampling or VAE-based, at either balancing level)
  produced a statistically significant improvement over the original
  imbalanced baseline.** That is a complete, valid answer to the research
  question -- not an incomplete result.
- If exactly one condition (e.g. C1 but not C2/C3) shows significant
  improvement, that tells you the improvement came from balancing itself,
  not specifically from synthetic image generation.
- Repeat SESSION 2 and SESSIONS 5/6 (C0, C1, C2, C3) with `SEED = 29` and
  `SEED = 41` before treating any single-seed pattern as final -- a
  difference that holds across all 3 seeds is real signal; one that
  doesn't is likely noise.
